# Lab 9: Detekcja anomalii w czasie rzeczywistym

## Cel

- Wytrenowanie modelu detekcji anomalii (Isolation Forest),
- Integracja modelu z pipeline'em strumieniowym,
- Kompletny system: źródło → Kafka → Spark → ML → alert,
- Mikroserwis FastAPI z modelem ML.

---

## Część 1: Trenowanie modelu offline

### Zadanie 1.1 — Przygotuj dane treningowe

Wygeneruj sztuczne dane transakcji (normalne + anomalie).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
import pickle

np.random.seed(42)

# Normalne transakcje: kwota 50-500, czestotliwosc 5-15/tydzien
normalne = pd.DataFrame({
    'kwota': np.random.normal(200, 80, 1000).clip(20, 800),
    'czestotliwosc': np.random.normal(10, 3, 1000).clip(1, 25),
    'godzina': np.random.normal(14, 4, 1000).clip(6, 23),
})
normalne['label'] = 0

# Anomalie: duze kwoty, nietypowe godziny, rzadkie
anomalie = pd.DataFrame({
    'kwota': np.random.uniform(3000, 10000, 50),
    'czestotliwosc': np.random.uniform(0, 2, 50),
    'godzina': np.random.uniform(0, 5, 50),
})
anomalie['label'] = 1

df = pd.concat([normalne, anomalie], ignore_index=True)
print(f"Dane: {len(df)} wierszy, anomalii: {df['label'].sum()}")

### Zadanie 1.2 — Wytrenuj Isolation Forest

Wytrenuj model na danych **bez etykiet** (metoda nienadzorowana).

In [ ]:
features = ['kwota', 'czestotliwosc', 'godzina']
X = df[features].values

# TWÓJ KOD
# model = IsolationForest(contamination=0.05, random_state=42)
# model.fit(X)

### Zadanie 1.3 — Ewaluacja modelu

Sprawdź, jak model radzi sobie z wykrywaniem anomalii (porównaj predykcje z etykietami).

In [ ]:
from sklearn.metrics import classification_report

# TWÓJ KOD
# predictions = model.predict(X)
# Zamien: -1 -> 1 (anomalia), 1 -> 0 (normalna)
# Wyswietl classification_report

### Zadanie 1.4 — Zapisz model

In [ ]:
# TWÓJ KOD
# pickle.dump(model, open('anomaly_model.pkl', 'wb'))

---

## Część 2: Pipeline strumieniowy

### Zadanie 2.1 — Producent transakcji

Napisz producenta Kafka, który wysyła losowe transakcje (w tym 5% anomalii) do tematu `transactions`.

In [ ]:
%%file tx_producer.py
from confluent_kafka import Producer
import json, random, time
from datetime import datetime
import numpy as np

conf = {'bootstrap.servers': 'localhost:9092'}
producer = Producer(conf)

def generate_transaction():
    is_anomaly = random.random() < 0.05
    if is_anomaly:
        return {
            'tx_id': f'TX{random.randint(10000,99999)}',
            'kwota': round(random.uniform(3000, 10000), 2),
            'czestotliwosc': round(random.uniform(0, 2), 1),
            'godzina': round(random.uniform(0, 5), 1),
            'timestamp': datetime.now().isoformat(),
        }
    else:
        # TWÓJ KOD - normalna transakcja
        pass

# TWÓJ KOD
# Petla wysylajaca transakcje do tematu 'transactions'
# Co 0.5 sekundy

### Zadanie 2.2 — Konsument z modelem ML

Napisz konsumenta Kafka, który:
1. Odczytuje transakcje z tematu `transactions`
2. Wczytuje model z `anomaly_model.pkl`
3. Klasyfikuje każdą transakcję
4. Wyświetla ALERT dla anomalii

In [ ]:
%%file fraud_detector.py
from confluent_kafka import Consumer
import json, pickle
import numpy as np

# Wczytaj model
model = pickle.load(open('anomaly_model.pkl', 'rb'))

conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'fraud-detection',
    'auto.offset.reset': 'earliest'
}

# TWÓJ KOD
# Subskrybuj temat 'transactions'
# W petli: pobieraj wiadomosci, parsuj JSON
# Klasyfikuj za pomoca modelu
# Wyswietlaj ALERT jesli anomalia

---

## Część 3: Mikroserwis FastAPI

### Zadanie 3.1 — API do klasyfikacji transakcji

Utwórz API, które przyjmuje dane transakcji i zwraca predykcję: fraud/not fraud.

In [ ]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np

app = FastAPI()
model = pickle.load(open('anomaly_model.pkl', 'rb'))

class Transaction(BaseModel):
    kwota: float
    czestotliwosc: float
    godzina: float

# TWÓJ KOD
# Endpoint POST /classify
# Przyjmuje Transaction, zwraca {"is_fraud": True/False, "score": ...}

### Zadanie 3.2 — Konteneryzacja

Utwórz Dockerfile dla API i dodaj usługę do docker-compose.yml.

In [ ]:
%%file Dockerfile.api
# TWÓJ KOD
# FROM python:3.11-slim
# ...

---

## Praca domowa

1. Uruchom pełny pipeline: producent → Kafka → konsument z modelem → alerty.
2. Dodaj zapis alertów do pliku CSV.
3. Skonteneryzuj całość w Docker Compose (Kafka + API + producent).
4. Wyślij kod do repozytorium Git.

Na następnych zajęciach: projekt końcowy.